<a href="https://colab.research.google.com/github/dhanushmajji1428/FlyRank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhanushmajji1428/FlyRank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df = df.fillna(0)

df["target"] = (
    (df["trend_direction"] == "down") &
    (df["avg_position"] > 20) &
    (df["impressions_90d"] > 1000)
).astype(int)

print(df[["target"]].head())

   target
0       0
1       1
2       1
3       0
4       1


## Method Choice

I selected a Decision Tree Classifier because it is simple, interpretable, and works well with structured tabular data. Unlike the Week 4 rule-based baseline, the Decision Tree learns patterns from multiple features automatically. It also provides feature importance, making it easy to understand which variables influence the predictions.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [20]:
X = df.select_dtypes(include=["number"]).drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 24000
Testing samples: 6000


## Split Design

I used an 80/20 train-test split with a fixed random state (42) and stratification to preserve the class distribution. The same split was used for both the baseline rule and the Decision Tree model to ensure a fair comparison and prevent data leakage.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [21]:
model = DecisionTreeClassifier(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

model_accuracy = accuracy_score(y_test, pred)

baseline_pred = (
    (X_test["trend_pct"] < 0) &
    (X_test["avg_position"] > 20) &
    (X_test["impressions_90d"] > 1000)
).astype(int)

baseline_accuracy = accuracy_score(y_test, baseline_pred)

comparison = pd.DataFrame({
    "Model": ["Baseline Rule", "Decision Tree"],
    "Accuracy": [baseline_accuracy, model_accuracy]
})

comparison

,Model,Accuracy
0,Baseline Rule,0.981833
1,Decision Tree,0.999833


## Model Comparison

The Decision Tree model was trained using the training data and evaluated on the same test set as the baseline.

| Model | Accuracy |
|--------|----------|
| Baseline Rule | 98.18% |
| Decision Tree | 99.98% |

The Decision Tree achieved higher accuracy than the baseline by learning patterns from multiple features rather than relying only on fixed thresholds.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [22]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("Top 10 Important Features")
print(importance.head(10))

errors = X_test.copy()
errors["Actual"] = y_test.values
errors["Predicted"] = pred

print("\nMisclassified Samples:")
display(errors[errors["Actual"] != errors["Predicted"]].head())

Top 10 Important Features
            Feature  Importance
29        trend_pct    0.400822
5   impressions_90d    0.399828
25     avg_position    0.199350
2               cpc    0.000000
1       competition    0.000000
0     search_volume    0.000000
6        clicks_90d    0.000000
7     pageviews_90d    0.000000
8      sessions_90d    0.000000
9         users_90d    0.000000

Misclassified Samples:


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,age_tier_order,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,trend_pct,Actual,Predicted
6972,60500.0,0.11,0.5,0.0,0.0,3141,0,15,13,13,...,6,104,0.0,45.5,0.0,13.33,0.0,-20.0,1,0


## Error Analysis

The Decision Tree correctly classified almost all samples, achieving an accuracy of 99.98%. Only one sample was misclassified. The most important features were `trend_pct`, `impressions_90d`, and `avg_position`, while the remaining features had little or no impact on the predictions. The single error occurred because the sample was close to the decision boundary, making it difficult to classify correctly.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.